# Baseline v6 — BM25 + Legal_HF Hybrid Retrieval

```
Pipeline v6:
  Query
    ├─ BM25 (lexical)    → top-100 ─┐
    │                               ├─ RRF merge → top-50 → CE (D_skip14) → rank #1
    └─ Legal_HF (dense) → top-100 ─┘
```

**Reciprocal Rank Fusion:**
```
score(doc) = 1/(k + rank_BM25) + 1/(k + rank_LegalHF)   # k=60
```

BM25 giỏi: số điều, khoản, tên văn bản cụ thể  
Legal_HF giỏi: ngữ nghĩa, paraphrase  
→ Kết hợp bổ sung nhau, tăng ceiling

**Target to beat:**  
v5+CE: R@1=0.5418, R@3=0.6873, R@5=0.7245, MRR=0.6307

## Cell 0 — Cài đặt & Imports

In [ ]:
import subprocess, sys
try:
    import rank_bm25
    print("rank-bm25 already installed ✓")
except ImportError:
    print("Installing rank-bm25...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rank-bm25", "-q"])
    print("rank-bm25 installed ✓")

In [ ]:
import json, csv, time, re
import numpy as np, faiss, torch
from pathlib import Path
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

ROOT         = Path(".")
DATA_DIR     = ROOT / "data"
EVAL_DIR     = ROOT / "outputs" / "eval"
TMP_DIR      = ROOT / "outputs" / "tmp"
MDL_DIR      = ROOT / "outputs" / "models"

TRAIN_FILE   = DATA_DIR / "train.jsonl"
DEV_FILE     = DATA_DIR / "dev.jsonl"
TRAIN_NEG    = DATA_DIR / "train_with_neg.jsonl"
EVAL_QA_FILE = EVAL_DIR / "eval_qa.jsonl"

# v4 fine-tuned bi-encoder
FT_BI_PATH   = MDL_DIR / "legal_hf_finetuned" / "final"
FAISS_V4     = TMP_DIR / "faiss_v4.index"
MAP_V4       = TMP_DIR / "faiss_mapping_v4.jsonl"

# CE v5 (D_skip14, best reranker)
CE_V5_PATH   = MDL_DIR / "cross_encoder_v5fix" / "saved_model"
if not CE_V5_PATH.exists():
    CE_V5_PATH = MDL_DIR / "cross_encoder_v5fix"

RERANK_CSV_V5= EVAL_DIR / "rerank_metrics_v5.csv"
RERANK_CSV_V6= EVAL_DIR / "rerank_metrics_v6.csv"

# ── Config ──
BM25_TOP_K   = 100    # BM25 retrieve top-100
DENSE_TOP_K  = 100    # Legal_HF retrieve top-100
RRF_K        = 60     # RRF constant (standard=60)
FINAL_TOP_N  = 50     # sau RRF, lấy top-50 cho CE
CE_BATCH     = 32

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"torch   : {torch.__version__}")
print(f"Device  : {DEVICE}")
print(f"BM25_K  : {BM25_TOP_K} | Dense_K: {DENSE_TOP_K} | RRF_K: {RRF_K}")
print(f"CE path : {CE_V5_PATH}")

## Cell 1 — Utilities

In [ ]:
def load_jsonl(path):
    rows, err = [], 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except: err += 1
    if err: print(f"  ⚠ {err} errors")
    return rows

def is_hit(corpus_id, ec, corpus):
    row = corpus[corpus_id]
    for e in ec:
        ci = e.get("chunk_index",-2)
        if ci!=-1 and row["chunk_index"]==ci: return True
        if (row["van_ban"]==e.get("van_ban","") and
            row["dieu"]   ==e.get("dieu",   "") and
            row["khoan"]  ==e.get("khoan",  "")): return True
    return False

def tokenize_vi(text):
    """Tokenize đơn giản: lowercase + split, giữ số + chữ."""
    text = text.lower()
    tokens = re.findall(r'[a-záàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵđ0-9]+', text)
    return tokens if tokens else text.split()

def rrf_merge(bm25_ids, dense_ids, k=60):
    """Reciprocal Rank Fusion: kết hợp 2 danh sách ranked."""
    scores = {}
    for rank, doc_id in enumerate(bm25_ids, 1):
        scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank)
    for rank, doc_id in enumerate(dense_ids, 1):
        scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def avg(lst): return round(sum(lst)/len(lst),4) if lst else 0.0

print("Utilities ✓")

## Cell 2 — Build Corpus + BM25 Index
> BM25 dùng tokenized text. Tiếng Việt tách whitespace là đủ cho legal text.

In [ ]:
# Thu thập corpus duy nhất
seen_passages = {}
for f in [TRAIN_FILE, DEV_FILE, TRAIN_NEG]:
    for r in load_jsonl(f):
        p = r.get("passage","")
        if p and p not in seen_passages:
            meta = r.get("meta",{})
            seen_passages[p] = {
                "passage":     p,
                "chunk_index": meta.get("chunk_index",-1),
                "van_ban":     meta.get("van_ban", ""),
                "chuong":      meta.get("chuong",  ""),
                "dieu":        meta.get("dieu",    ""),
                "khoan":       meta.get("khoan",   ""),
                "diem":        meta.get("diem",    ""),
            }

corpus = list(seen_passages.values())
print(f"Corpus: {len(corpus)} unique passages")

# Tokenize cho BM25
print("Tokenizing corpus for BM25...")
t0 = time.perf_counter()
tokenized = [tokenize_vi(c["passage"]) for c in tqdm(corpus, desc="Tokenize")]
print(f"  Avg tokens/passage: {sum(len(t) for t in tokenized)/len(tokenized):.0f}")

# Build BM25
print("Building BM25 index...")
bm25 = BM25Okapi(tokenized)
print(f"BM25 index built in {time.perf_counter()-t0:.1f}s ✓")

## Cell 3 — Load Legal_HF FAISS (v4) + CE (v5 D_skip14)

In [ ]:
print("Loading v4 fine-tuned bi-encoder...")
bi_model = SentenceTransformer(str(FT_BI_PATH), device=DEVICE)

# Load FAISS v4
print("Loading FAISS v4...")
index_v4   = faiss.read_index(str(FAISS_V4))
mapping_v4 = load_jsonl(MAP_V4)

# Build passage→corpus_id lookup (mapping_v4 có cùng passages với corpus)
passage_to_cid = {c["passage"]: i for i, c in enumerate(corpus)}
faiss_to_cid   = {}   # faiss_id → corpus_id
for entry in mapping_v4:
    fid = entry["faiss_id"]
    p   = entry["passage"]
    if p in passage_to_cid:
        faiss_to_cid[fid] = passage_to_cid[p]

print(f"  Bi-encoder ✓ | FAISS: {index_v4.ntotal} vectors")
print(f"  faiss→corpus_id mapping: {len(faiss_to_cid)} entries")

# Load CE v5 (D_skip14)
print(f"Loading CE v5: {CE_V5_PATH}")
ce_model = CrossEncoder(str(CE_V5_PATH), max_length=256, device=DEVICE)
print("All models loaded ✓")

## Cell 4 — Evaluate v6: BM25 only / Dense only / Hybrid / Hybrid+CE

Chạy **4 variants** cùng lúc để so sánh contribution của từng component:

In [ ]:
eval_qa = load_jsonl(EVAL_QA_FILE)
print(f"Eval QA: {len(eval_qa)} questions")

# Kết quả cho 4 variants
variants = {
    "BM25_only":   {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]},
    "Dense_only":  {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]},
    "Hybrid_RRF":  {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]},
    "Hybrid+CE":   {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]},
}

def eval_ranks(ids_, ec, res_dict, key):
    for k, rk in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        res_dict[rk].append(1 if any(is_hit(i,ec,corpus) for i in ids_[:k]) else 0)
    mrr=0.0
    for rank,i in enumerate(ids_[:10],1):
        if is_hit(i,ec,corpus): mrr=1.0/rank; break
    res_dict["MRR@10"].append(mrr)

for item in tqdm(eval_qa, desc="Evaluate v6"):
    query = item["query"]; ec = item["expected_citations"]

    # ── BM25 retrieve ──
    q_tokens  = tokenize_vi(query)
    bm25_scores = bm25.get_scores(q_tokens)
    bm25_ids  = np.argsort(bm25_scores)[::-1][:BM25_TOP_K].tolist()

    # ── Dense retrieve ──
    q_emb = bi_model.encode([query], normalize_embeddings=True,
                             convert_to_numpy=True).astype("float32")
    _, faiss_ids = index_v4.search(q_emb, DENSE_TOP_K)
    dense_cids   = [faiss_to_cid[fid] for fid in faiss_ids[0].tolist()
                    if fid>=0 and fid in faiss_to_cid]

    # ── RRF Merge ──
    merged = rrf_merge(bm25_ids, dense_cids, k=RRF_K)
    hybrid_ids = [doc_id for doc_id, _ in merged[:FINAL_TOP_N]]

    # ── Eval: BM25 only ──
    eval_ranks(bm25_ids[:FINAL_TOP_N], ec, variants["BM25_only"], "")

    # ── Eval: Dense only ──
    eval_ranks(dense_cids[:FINAL_TOP_N], ec, variants["Dense_only"], "")

    # ── Eval: Hybrid RRF (no CE) ──
    eval_ranks(hybrid_ids, ec, variants["Hybrid_RRF"], "")

    # ── Eval: Hybrid + CE rerank ──
    cands   = [(corpus[i]["passage"], i) for i in hybrid_ids]
    rscores = ce_model.predict([[query,c[0]] for c in cands], batch_size=CE_BATCH) if cands else []
    ranked  = sorted(zip(rscores,[c[1] for c in cands]),reverse=True)
    r_ids   = [x[1] for x in ranked]
    eval_ranks(r_ids, ec, variants["Hybrid+CE"], "")

print("\n── v6 Results ──")
print(f"  {'Variant':<14} {'R@1':>8} {'R@3':>8} {'R@5':>8} {'MRR@10':>8}")
print("  " + "-"*46)
for vname, vres in variants.items():
    print(f"  {vname:<14} {avg(vres['R@1']):>8.4f} {avg(vres['R@3']):>8.4f} {avg(vres['R@5']):>8.4f} {avg(vres['MRR@10']):>8.4f}")

## Cell 5 — So sánh v5 vs v6 & Lưu CSV

In [ ]:
# Kết quả v5 đã biết (D_skip14)
V5 = {"R@1":0.5418,"R@3":0.6873,"R@5":0.7245,"MRR@10":0.6307}
V6 = {
    "R@1":   avg(variants["Hybrid+CE"]["R@1"]),
    "R@3":   avg(variants["Hybrid+CE"]["R@3"]),
    "R@5":   avg(variants["Hybrid+CE"]["R@5"]),
    "MRR@10":avg(variants["Hybrid+CE"]["MRR@10"]),
}

print("\n" + "="*80)
print(f"  {'Metric':<10} {'v5 (D_skip14)':>16} {'v6 (Hybrid+CE)':>16} {'Δ':>10} {'Win':>5}")
print("="*80)
for m in ["R@1","R@3","R@5","MRR@10"]:
    d = V6[m] - V5[m]
    win = "✅" if d > 0.001 else ("❌" if d < -0.001 else "=")
    print(f"  {m:<10} {V5[m]:>16.4f} {V6[m]:>16.4f} {d:>+9.4f} {win:>5}")
print("="*80)

# Breakdown: BM25 vs Dense vs Hybrid
print("\n── Breakdown components ──")
print(f"  {'Variant':<14} {'R@1':>8} {'R@3':>8} {'R@5':>8} {'MRR':>8}")
print("  " + "-"*46)
for vname, vres in variants.items():
    r1=avg(vres['R@1']); r3=avg(vres['R@3'])
    r5=avg(vres['R@5']); mrr=avg(vres['MRR@10'])
    marker = " ★" if vname=="Hybrid+CE" else ""
    print(f"  {vname:<14} {r1:>8.4f} {r3:>8.4f} {r5:>8.4f} {mrr:>8.4f}{marker}")
print(f"  {'v5 (baseline)':14} {V5['R@1']:>8.4f} {V5['R@3']:>8.4f} {V5['R@5']:>8.4f} {V5['MRR@10']:>8.4f}")

# Lưu CSV đầy đủ
rows_v6 = []
for metric in ["R@1","R@3","R@5","MRR@10"]:
    rows_v6.append({
        "metric":      metric,
        "v5_D_skip14": V5[metric],
        "v6_bm25_only":   avg(variants["BM25_only"][metric if metric!="MRR@10" else "MRR@10"]),
        "v6_dense_only":  avg(variants["Dense_only"][metric if metric!="MRR@10" else "MRR@10"]),
        "v6_hybrid_rrf":  avg(variants["Hybrid_RRF"][metric if metric!="MRR@10" else "MRR@10"]),
        "v6_hybrid_ce":   V6[metric],
    })
# sửa key cho MRR
metric_key = lambda m: "MRR@10" if m=="MRR@10" else m
rows_v6 = []
for metric in ["R@1","R@3","R@5","MRR@10"]:
    mk = metric_key(metric)
    rows_v6.append({
        "metric":          metric,
        "v5_D_skip14":     V5[metric],
        "v6_bm25_only":    avg(variants["BM25_only"][mk]),
        "v6_dense_only":   avg(variants["Dense_only"][mk]),
        "v6_hybrid_rrf":   avg(variants["Hybrid_RRF"][mk]),
        "v6_hybrid_ce":    V6[metric],
    })

with open(RERANK_CSV_V6,"w",newline="",encoding="utf-8") as f:
    w = csv.DictWriter(f,fieldnames=["metric","v5_D_skip14","v6_bm25_only","v6_dense_only","v6_hybrid_rrf","v6_hybrid_ce"])
    w.writeheader(); w.writerows(rows_v6)
print(f"\nSaved → {RERANK_CSV_V6} ✓")